# Autograd Deep Dive: `loss = (a*x - y)**2`

We trace the computational graph built by PyTorch's autograd engine.

- `a` = learnable parameter (`requires_grad=True`)
- `x` = input data (`requires_grad=False`)
- `y` = target data (`requires_grad=False`)

We will inspect `grad_fn`, `next_functions`, and saved tensors at every node.

In [ ]:
import torch

print("torch version:", torch.__version__)

# Parameter: we need gradients w.r.t. this
a = torch.tensor(2.0, requires_grad=True)

# Data: no gradients needed
x = torch.tensor(3.0)
y = torch.tensor(1.0)

print(f"a = {a.item()}, requires_grad={a.requires_grad}, is_leaf={a.is_leaf}")
print(f"x = {x.item()}, requires_grad={x.requires_grad}, is_leaf={x.is_leaf}")
print(f"y = {y.item()}, requires_grad={y.requires_grad}, is_leaf={y.is_leaf}")

## Leaves

`a`, `x`, `y` are all **leaf tensors** (`is_leaf=True`) because they are user-created.

But only `a` has `requires_grad=True`, so only `a` will appear as a node in the backward graph.

Leaves have `grad_fn = None` — they are graph *inputs*, not outputs of an operation.

In [ ]:
print("a.grad_fn:", a.grad_fn)
print("x.grad_fn:", x.grad_fn)
print("y.grad_fn:", y.grad_fn)

## Forward pass, step 1: `t1 = a * x`

The multiplication creates a `MulBackward0` node.

Because `x` has `requires_grad=False`, PyTorch does **not** create a graph slot for it.  
`next_functions` will have **only one** entry — for `a`.

`MulBackward0` still **saves `x` internally** because it needs `x` to compute `∂t1/∂a`.

In [ ]:
t1 = a * x
print(f"t1 = {t1.item()}")
print(f"t1.grad_fn = {t1.grad_fn}")
print(f"t1.grad_fn.next_functions = {t1.grad_fn.next_functions}")

## Forward pass, step 2: `t2 = t1 - y`

Subtraction creates a `SubBackward0` node.

Again, `y` has `requires_grad=False`, so only **one** slot in `next_functions`: for `t1`.

`SubBackward0` doesn't need to save anything — the backward of `+` / `-` is a constant (`+1` / `-1`).

In [ ]:
t2 = t1 - y
print(f"t2 = {t2.item()}")
print(f"t2.grad_fn = {t2.grad_fn}")
print(f"t2.grad_fn.next_functions = {t2.grad_fn.next_functions}")

## Forward pass, step 3: `loss = t2 ** 2`

Exponentiation creates a `PowBackward0` node.

The exponent here is the Python **int** `2`, not a tensor.  
So `PowBackward0` has only **one** graph input: the base `t2`.

`PowBackward0` saves:
- `self` = `t2` (the base, 5.0)
- `exponent` = 2

In [ ]:
loss = t2 ** 2
print(f"loss = {loss.item()}")
print(f"loss.grad_fn = {loss.grad_fn}")
print(f"loss.grad_fn.next_functions = {loss.grad_fn.next_functions}")

## The full graph

```
a ──┐
    ├── MulBackward0 ── t1 ── SubBackward0 ── t2 ── PowBackward0 ── loss
x ──┘                         y ──┘
(data, no grad)               (data, no grad)
```

Only `a` has a gradient path back.  
`x` and `y` are consumed in forward but are **not** nodes in the backward graph.

The `None` in `next_functions` marks a leaf tensor — here, `a`.

## Peek inside saved tensors (optional, private API)

Each `*Backward` node saves what it needs for backward.  
The field names (`_saved_self`, `_saved_other`, ...) are op-specific and not public API — but they reveal what's retained.

In [ ]:
# PowBackward0 saves the base and the exponent
print("loss.grad_fn._saved_self     =", loss.grad_fn._saved_self)
print("loss.grad_fn._saved_exponent =", loss.grad_fn._saved_exponent)

# MulBackward0 saves both operands (a and x) even though x doesn't need grad
print("t1.grad_fn._saved_self  =", t1.grad_fn._saved_self)
print("t1.grad_fn._saved_other =", t1.grad_fn._saved_other)

## Backward pass

Calling `loss.backward()` walks the graph in reverse topological order:

1. `PowBackward0`: incoming grad 1 → outgoing `2 * t2 = 10` to `SubBackward0`
2. `SubBackward0`: incoming 10 → outgoing 10 to `MulBackward0` (no `y` slot)
3. `MulBackward0`: incoming 10 → outgoing `10 * x = 30` into `a.grad`

`x.grad` and `y.grad` remain `None` — they were never graph leaves with `requires_grad=True`.

In [ ]:
loss.backward()

print("a.grad =", a.grad)          # 30.0
print("x.grad =", x.grad)          # None
print("y.grad =", y.grad)          # None

## Manual verification

$$
\text{loss} = (a x - y)^2
$$

$$
\frac{\partial \text{loss}}{\partial a} = 2(ax - y) \cdot x
$$

With `a=2`, `x=3`, `y=1`:

$$
2 \cdot (2 \cdot 3 - 1) \cdot 3 = 2 \cdot 5 \cdot 3 = 30
$$

✅ Matches `a.grad`.

## Typical training loop

Now we wrap this in a proper optimization loop to see `a` move toward its minimizer.

The true minimum is at `a* = y / x = 1/3`.

In [ ]:
a = torch.tensor(2.0, requires_grad=True)
optimizer = torch.optim.SGD([a], lr=0.01)

x = torch.tensor(3.0)
y = torch.tensor(1.0)

for step in range(5):
    pred = a * x
    loss = (pred - y) ** 2

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(
        f"step {step}: "
        f"a = {a.item():.4f}, "
        f"a.grad = {a.grad.item():.4f}, "
        f"loss = {loss.item():.4f}"
    )

## Key takeaways

1. `grad_fn` is a **node** in a DAG rebuilt on every forward pass.
2. `next_functions` is a tuple of `(grad_fn, input_index)` pairs.
3. A `None` entry marks a **leaf tensor**.
4. Tensors with `requires_grad=False` (`x`, `y`) **do not appear** in `next_functions`.
5. `MulBackward0` still **saves `x`** internally, because backward needs `x` to compute `∂loss/∂a`.
6. Saving a tensor for backward is **independent** of that tensor requiring a gradient.
7. Only `a` — the parameter — accumulates a gradient in `.grad`.